In [7]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import joblib

# =========================
# 2. LOAD DATASET
# =========================
# IMPORTANT: agar error aaye to path change karo
df = pd.read_csv("Telco-Customer-Churn.csv")

# =========================
# 3. DATA CLEANING
# =========================
# Drop unnecessary column
df.drop("customerID", axis=1, inplace=True)

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Convert target to numeric
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# =========================
# 4. SPLIT FEATURES
# =========================
X = df.drop("Churn", axis=1)
y = df["Churn"]

# =========================
# 5. COLUMN TYPES
# =========================
categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns

# =========================
# 6. PREPROCESSING
# =========================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# =========================
# 7. MODEL + PIPELINE
# =========================
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# =========================
# 8. TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 9. GRID SEARCH (IMPORTANT)
# =========================
param_grid = {
    "model__C": [0.1, 1, 10],
    "model__solver": ["liblinear"]
}

grid = GridSearchCV(pipeline, param_grid, cv=3, scoring="accuracy")

# TRAIN MODEL
grid.fit(X_train, y_train)

# =========================
# 10. EVALUATION
# =========================
y_pred = grid.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# =========================
# 11. SAVE MODEL (IMPORTANT)
# =========================
joblib.dump(grid, "churn_pipeline.pkl")

print("\nMODEL SAVED SUCCESSFULLY ✅")

Accuracy: 0.8204400283889283

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.90      0.88      1036
           1       0.69      0.60      0.64       373

    accuracy                           0.82      1409
   macro avg       0.77      0.75      0.76      1409
weighted avg       0.81      0.82      0.82      1409


MODEL SAVED SUCCESSFULLY ✅


In [11]:
import joblib

joblib.dump(grid, "churn_pipeline.pkl")

['churn_pipeline.pkl']

In [12]:
import os
print(os.path.expanduser("~"))

C:\Users\HP


In [13]:
joblib.dump(grid, r"C:\Users\HP\OneDrive\Desktop\churn_pipeline.pkl")

['C:\\Users\\HP\\OneDrive\\Desktop\\churn_pipeline.pkl']